# CAD primitive learning

In this notebook I will explore the second step of my proposed idea, where PN++ and Deepcad are trained together to learn extrusion primitives.

**IMPORTANT** 
For this I again turned of the random sampling in the dataset, in order to get all points.

## Dataset creation

In [9]:
import os
import shutil
import h5py
import numpy as np
import sys
import pandas as pd
import json
import matplotlib.pyplot as plt
%matplotlib inline

sys.path.append("..")
sys.path.append("../code")

from dataset import PCExtrusionSegmentationDataset, BaseDataset

import os
import shutil
import h5py
import numpy as np
import sys
import pandas as pd
import json
import matplotlib.pyplot as plt
%matplotlib inline

sys.path.append("..")
sys.path.append("../code")

import open3d as o3d
import torch

In [2]:
train_dataset = PCExtrusionSegmentationDataset("../data", 'train', use_normals=False, verbose=False)
val_dataset = PCExtrusionSegmentationDataset("../data", 'validation', use_normals=False, verbose=False)
test_dataset = PCExtrusionSegmentationDataset("../data", 'test', use_normals=False, verbose=False)
datasets = [train_dataset, val_dataset, test_dataset]

In [3]:
def save_pc(pc, path):
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(pc)
    o3d.io.write_point_cloud(path, pcd)

In [4]:
def save_h5(extr_id, sequence, path):
    with h5py.File(path, "w") as f:
        f.create_dataset("extrusion_id", data=extr_id)
        f.create_dataset("sequence", data=sequence)

In [8]:
DATA_DIR = "data_exp"
error_dict = {}

for dataset in datasets:
    length = len(dataset)
    
    for i in range(length):
        
        try:
            data = dataset[i]
            id = data['id']
            print(id)
            pc = data['pc']
            label = data['label']
    
            pcs = split_pc_by_labels(pc, label)
    
            sequences = {}
            h5_path = os.path.join("..", "data", "pc_from_vec_labels", id[:4], id + ".h5") # Replace ../data with DATA_DIR
            with h5py.File(h5_path, 'r') as f:
                for class_id, sequence in f['sequences'].items():
                    sequence = sequence[:]
                    seq_length = np.where(sequence[:, 0] == 3)[0][0] + 1 # only save up to including the first EOS command
                    sequence = sequence[:seq_length, :]
                    sequences[class_id] = sequence
            
            for i, pc in enumerate(pcs):
                pc_path = os.path.join(os.path.abspath(DATA_DIR), "pc_extrusion", id[:4], id, id + "_" + str(i) + ".ply")
                os.makedirs(os.path.dirname(pc_path), exist_ok=True)
                save_pc(pc, pc_path)
                
                h5_path = os.path.join(os.path.abspath(DATA_DIR), "pc_extrusion_labels", id[:4], id, id + "_" + str(i) + ".h5")
                os.makedirs(os.path.dirname(h5_path), exist_ok=True)
                sequence = sequences[str(i)]
                save_h5(i, sequence, h5_path)
                
        except Exception as e:
            error_dict[i] = e


        if i == 1:
            break
    break

00675619
00981499
00435622
00020470
00796013
00034076


In [188]:
def sanity_check(id):
    gt_pc_path = os.path.join("..", "data", "pc_from_vec", id[:4], id + ".ply")
    gt_lable_path = os.path.join("..", "data", "pc_from_vec_labels", id[:4], id + ".h5")

    gt_pc = o3d.io.read_point_cloud(gt_pc_path)
    gt_pc = np.asarray(gt_pc.points)

    with h5py.File(gt_lable_path, "r") as f:
        print(f.keys())
        gt_label = f['labels'][:]
    
    visualize_labeled_pc(gt_pc, gt_label)
    

    
    
    pc_dir = os.path.join(DATA_DIR, "pc_extrusion", id[:4], id)
    pc_paths = sorted([os.path.join(pc_dir, fname) for fname in os.listdir(pc_dir)])
    h5_dir = os.path.join(os.path.abspath(DATA_DIR), "pc_extrusion_labels", id[:4], id)
    h5_paths = sorted([os.path.join(h5_dir, fname) for fname in os.listdir(h5_dir)])

    for pc_file, h5_file in zip(pc_paths, h5_paths):
        point_cloud = o3d.io.read_point_cloud(pc_file)
        point_cloud = np.asarray(point_cloud.points)
        
        with h5py.File(h5_file, "r") as f:
            extr_id = f['extrusion_id'][()]
            sequence = f['sequence'][:]
        print(extr_id)
        print(sequence)
        print()
        
        visualize_pc(point_cloud)

In [193]:
sanity_check("00034076")

<KeysViewHDF5 ['labels', 'sequences']>
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
0
[[  4  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1]
 [  2 176 128  -1  -1  48  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1]
 [  4  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1]
 [  2 176 128  -1  -1  25  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1]
 [  5  -1  -1  -1  -1  -1 192  64 192  98 128 128  60 141 128   0   0]
 [  3  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1]]

[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
1
[[  4  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1]
 [  2 176 128  -1  -1  48  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1]
 [  4  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1]
 [  2 176 12

In [47]:
from glob import glob
class PCExtrusionSequenceDataset(BaseDataset):
    def __init__(self, root, split, use_normals=False, verbose=False):
        super().__init__(root, split, use_normals=use_normals, verbose=verbose)
        self.pc_path = os.path.join(root, "pc_extrusion")

        # Scrutinize if the id's mentioned in the split actually exist as directories
        valid_dirs = []
        pc_all = self.read_split()
        for dir in pc_all:
            if os.path.isdir(dir):
                valid_dirs.append(dir)

        # Get all point clouds from valid directories
        self.pc = []
        for dir in valid_dirs:
            pc_paths = glob(os.path.join(dir, "*.ply"))
            self.pc.extend(pc_paths)

    def __getitem__(self, idx):
        pass

    def read_split(self):
        with open(self.split_path, "r") as fp:
            all_data = json.load(fp)
        if self.verbose:
            print(f"Number of samples that should be in the {self.split} set: {len(all_data[self.split])}", flush=True)
        pc_set = [os.path.join(self.pc_path, f"{idx}") for idx in all_data[self.split]]
        return pc_set

    def filter_pc(self, pc_set):
        pc_set_filtered = [entry for entry in pc_set if entry in self.all_pc_files]
        corrupt_files = [i for i, entry in enumerate(pc_set) if entry not in self.all_pc_files]
        if self.verbose:
            print(f"Files on disk: {len(pc_set_filtered)} --> There are {len(corrupt_files)} missing point cloud files in the {self.split} set.\n", flush=True)
        return pc_set_filtered, corrupt_files

In [36]:
a = PCExtrusionSequenceDataset("../data", "train", verbose=True)

Loading train dataset 

Number of samples that should be in the train set: 161240
Files on disk: 0 --> There are 161240 missing point cloud files in the train set.

Checking latent representation:
All latent represenations are valid.

--- DONE ---

Number of samples that should be in the train set: 161240
['../data/pc_extrusion/0067/00675619', '../data/pc_extrusion/0098/00981499', '../data/pc_extrusion/0043/00435622']
161240
252958
Files on disk: 0 --> There are 161240 missing point cloud files in the train set.



In [37]:
os.path.isdir('../data/pc_extrusion/0067/00675619')


True

In [41]:
a = [1,2,3]
b = [4,5,6]

In [42]:
a.extend(b)

In [43]:
a

[1, 2, 3, 4, 5, 6]

In [ ]:
NEXT: CHECK IF NUMBER OF EXTRUSION IN ENTIRE DATASET MATCHES NUMBER OF PC
ADD targets